# Drift Monitoring — Inference Monitoring Notebook

> 📊 **Best viewed on [nbviewer](https://nbviewer.org/)** — GitHub's renderer strips the JavaScript that powers Evidently's interactive drift reports.

End-to-end drift monitoring for the deployed endpoint. Infrastructure (SNS topic, SQS results queue, monitoring-results writer Lambda, CloudWatch dashboard + alarms, IAM roles) is provisioned by `cloudformation/drift-monitoring-infra.yaml` via the deploy cell in Section 1 — no imperative setup lives in this notebook.

You focus on the parts that need a data scientist:

1. **Setup** — deploy the drift-monitoring infrastructure stack, load config, confirm the resources
2. **Generate drifted data** *(dev/test only)* — fabricate traffic that should trip the detector
3. **Send predictions** — invoke the endpoint so there's something to monitor
4. **Apply ground truth** — simulate fraud-investigation labels for model-drift metrics
5. **Detect drift** — the core: data drift + model drift with Evidently, logged to MLflow & Athena
6. **Schedule it** *(optional)* — hand the same drift check to a daily Lambda

> **Rule**: never send predictions (Section 3) *after* running the ground-truth simulator (Section 4) — those rows would have NULL ground_truth and skew model-drift metrics. The cell order enforces this.

> The prior imperative version of this notebook is kept as `3_inference_monitoring_legacy.ipynb` for reference; it is deprecated in favor of the CFN-backed workflow here.

## Prerequisites

This notebook assumes the following already exist. If any are missing, deploy
them before continuing:

| Prerequisite | How it's created |
|---|---|
| SageMaker domain, S3 data bucket, Athena tables, MLflow server, inference-logging path | `cloudformation/sagemaker-mlflow-setup.yaml` (base stack) |
| A trained + **Approved** model in the `fraud-detection` model package group | `1_training_pipeline.ipynb` |
| A live endpoint (`fraud-detector-endpoint`) logging to `inference_responses` | `2_deployment.ipynb` |
| **Drift-monitoring plane** — SNS, SQS results queue, writer Lambda, dashboard, alarms, drift-monitor IAM role | `cloudformation/drift-monitoring-infra.yaml` (deployed by the cell in Section 1.1 below) |

The `.env` file (populated by the base stack's lifecycle script) already carries `DATA_S3_BUCKET`, `AWS_DEFAULT_REGION`, and `ALERT_EMAIL`. Section 1 loads them; the deploy cell that follows just references those values — nothing to hand-edit.

## 1. Setup

In [ ]:
# Install project dependencies into the kernel.
# Idempotent — fast on re-runs since pip detects everything's already installed.
# Required on first run of a fresh JupyterLab kernel (SageMaker base image
# doesn't ship with evidently / awswrangler / pyathena / etc).
!pip install -e ../. --quiet


In [ ]:
import sys, os, json, time, logging, uuid
from pathlib import Path
from datetime import datetime, timedelta

import boto3
import pandas as pd
import numpy as np

# Find project root and add to path
project_root = Path.cwd()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

from src.config.config import (
    ATHENA_DATABASE, DATA_S3_BUCKET, AWS_DEFAULT_REGION,
    ATHENA_OUTPUT_S3,
    ATHENA_TRAINING_TABLE, ATHENA_EVALUATION_TABLE,
    ATHENA_INFERENCE_TABLE,
    ATHENA_GROUND_TRUTH_UPDATES_TABLE,
    CSV_DRIFTED_DATA,
    SAGEMAKER_EXEC_ROLE, MLFLOW_MODEL_NAME,
    MONITORING_DATA_DRIFT_LOOKBACK_DAYS,
    GROUND_TRUTH_SIM_ACCURACY,
    GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT,
    GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
)
from src.config import schema
from src.train_pipeline.athena.athena_client import AthenaClient

TRAINING_FEATURES = schema.feature_names()
TARGET_COLUMN = schema.target_column()

# Data source policy:
#   - training_data, evaluation_data, monitoring_responses, inference_responses
#     are read from Athena (single source of truth across runs).
#   - drifted samples (cell 11) are read from the local CSV that cells 6/8
#     just wrote — they are a transient per-run artifact, not durable state.

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

ENDPOINT_NAME = 'fraud-detector-endpoint'
REGION = AWS_DEFAULT_REGION

runtime_client = boto3.client('sagemaker-runtime', region_name=REGION)
sm_client = boto3.client('sagemaker', region_name=REGION)
athena_client = AthenaClient()

print(f'Region: {REGION}')
print(f'Endpoint: {ENDPOINT_NAME}')
print(f'Athena DB: {ATHENA_DATABASE}')
print(f'S3 Bucket: {DATA_S3_BUCKET}')
print(f'MLflow URI: {os.getenv("MLFLOW_TRACKING_URI", "NOT SET")}')
print(f'Data Drift Lookback: {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days')


### 1.1 Deploy the drift-monitoring plane

Idempotent — safe to re-run. Reads config values already loaded above; override
`ALERT_EMAIL` in `.env` before running to receive real drift alerts.

In [ ]:
# Deploy the drift-monitoring CloudFormation stack (idempotent).
# Resource identifiers come from src.config.config (backed by .env), which
# the imports cell above already resolved — no manual editing needed.
# Override ALERT_EMAIL in .env for real alert delivery (default is a no-deliver
# example.com address).

ALERT_EMAIL_RESOLVED = os.getenv('ALERT_EMAIL', 'nobody@example.com')

print(f'Deploying drift-monitoring stack with:')
print(f'  Data bucket   : {DATA_S3_BUCKET}')
print(f'  Endpoint name : {ENDPOINT_NAME}')
print(f'  Alert email   : {ALERT_EMAIL_RESOLVED}')
print(f'  Region        : {AWS_DEFAULT_REGION}')
print()

!bash ../cloudformation/deploy-drift-monitoring.sh \
    --data-bucket {DATA_S3_BUCKET} \
    --endpoint-name {ENDPOINT_NAME} \
    --alert-email {ALERT_EMAIL_RESOLVED} \
    --region {AWS_DEFAULT_REGION}

### 1.2 Confirm infrastructure

Read-only. Resolves the drift-monitoring resources from the CloudFormation
stack outputs (or config/.env defaults) and defines the resource-name
constants the later cells use. Creates nothing.

In [ ]:
# ── Confirm the drift-monitoring infrastructure (provisioned by CloudFormation) ──
# This notebook does NOT create infrastructure. cloudformation/drift-monitoring-infra.yaml
# owns the SNS topic, SQS results queue, monitoring-results writer Lambda,
# CloudWatch dashboard + alarms, and the drift-monitor IAM role. This cell just
# reads those resource names (from the stack outputs when available, otherwise
# from config/.env defaults) and prints them so you can confirm they exist.

import boto3

# Name of the drift-monitoring infra stack (override in .env if you named it differently)
DRIFT_INFRA_STACK = os.getenv('DRIFT_INFRA_STACK_NAME', 'fraud-detection-drift-monitoring')

# Resource-name constants (defaults match cloudformation/drift-monitoring-infra.yaml).
# .env overrides win so a customized deployment still resolves.
DRIFT_LAMBDA_NAME        = os.getenv('DRIFT_LAMBDA_NAME', 'fraud-detection-drift-monitor')
MONITORING_WRITER_LAMBDA = os.getenv('MONITORING_WRITER_LAMBDA_NAME', 'fraud-monitoring-results-writer')
MONITORING_SQS_QUEUE_NAME = os.getenv('MONITORING_SQS_QUEUE_NAME', 'fraud-monitoring-results')
SNS_TOPIC_NAME           = os.getenv('SNS_TOPIC_NAME', 'fraud-detection-drift-alerts')
EVENTBRIDGE_RULE_NAME    = os.getenv('EVENTBRIDGE_RULE_NAME', 'fraud-detection-drift-check')
CLOUDWATCH_DASHBOARD_NAME = os.getenv('CLOUDWATCH_DASHBOARD_NAME', 'FraudDetection-DriftMonitoring')
CLOUDWATCH_NAMESPACE     = os.getenv('CLOUDWATCH_NAMESPACE', 'FraudDetection/DriftMonitoring')
CLOUDWATCH_LOG_GROUP_DRIFT = os.getenv('CLOUDWATCH_LOG_GROUP_DRIFT', f'/aws/lambda/{DRIFT_LAMBDA_NAME}')
MONITORING_TABLE_NAME    = os.getenv('MONITORING_TABLE_NAME', 'monitoring_responses')

# Library code (src.drift_monitoring.lambda_drift_monitor.load_baseline_from_registry)
# reads these directly, so export them to the process env.
os.environ.setdefault('ENDPOINT_NAME', ENDPOINT_NAME)
os.environ.setdefault('MODEL_PACKAGE_GROUP', 'fraud-detection')

# Best-effort: pull the live resource ARNs/URLs from the CloudFormation stack outputs.
stack_outputs = {}
try:
    cfn = boto3.client('cloudformation', region_name=REGION)
    stacks = cfn.describe_stacks(StackName=DRIFT_INFRA_STACK)['Stacks']
    stack_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
except Exception as e:
    print(f'ℹ Could not read stack "{DRIFT_INFRA_STACK}" outputs ({type(e).__name__}). '
          f'Using config/.env defaults for resource names.')

print('━' * 72)
print('DRIFT-MONITORING INFRASTRUCTURE (managed by CloudFormation)')
print('━' * 72)
print(f'  Infra stack        : {DRIFT_INFRA_STACK}'
      f'{"  ✓ found" if stack_outputs else "  (not queried — using defaults)"}')
print(f'  SNS alert topic    : {stack_outputs.get("DriftAlertsTopicArn", SNS_TOPIC_NAME)}')
print(f'  Results SQS queue  : {stack_outputs.get("MonitoringResultsQueueUrl", MONITORING_SQS_QUEUE_NAME)}')
print(f'  Writer Lambda      : {stack_outputs.get("MonitoringWriterFunctionArn", MONITORING_WRITER_LAMBDA)}')
print(f'  Drift-monitor role : {stack_outputs.get("DriftMonitorRoleArn", "fraud-detection-drift-monitor-role")}')
print(f'  CloudWatch board   : {stack_outputs.get("DashboardName", CLOUDWATCH_DASHBOARD_NAME)}')
print(f'  Monitoring table   : {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}')
print('━' * 72)
print('If the stack was NOT found, deploy it once before running Section 5:')
print('  aws cloudformation deploy \\')
print('    --template-file cloudformation/drift-monitoring-infra.yaml \\')
print(f'    --stack-name {DRIFT_INFRA_STACK} \\')
print('    --capabilities CAPABILITY_NAMED_IAM \\')
print(f'    --parameter-overrides DataBucketName={DATA_S3_BUCKET} EndpointName={ENDPOINT_NAME}')

## 2. Generate Drifted Test Data *(dev/test only)*

Fabricate a drifted CSV the detector should flag, so Section 5 has something to
find. In production you'd skip this entirely — real traffic supplies the drift.
Drift parameters live in `src/config/config.yaml` → `drift_generation.default_drift`.

In [ ]:
# Generate drifted dataset (configured via config.yaml)
import subprocess
import sys

print('=' * 80)
print('GENERATING DRIFTED TEST DATA')
print('=' * 80)
print('\nThis will create data/creditcard_drifted.csv with configurable drift amounts.')
print('Drift parameters are read from: src/config/config.yaml → drift_generation.default_drift\n')

result = subprocess.run(
    [sys.executable, 'src/drift_monitoring/generate_drift_dataset.py'],
    cwd=Path.cwd().parent,  # Run from project root
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error generating drifted dataset:')
    print(result.stderr)
else:
    print('\n✅ Drifted dataset generated successfully!')
    print('   File: data/creditcard_drifted.csv')
    print('   To use this data, send it to your endpoint for inference testing.')

## 3. Send Predictions to the Endpoint

Every cell here invokes the deployed endpoint. The custom handler auto-logs each
prediction to Athena via SQS → Lambda → `inference_responses` (batches of 10 or
every 30s). Most invocations send intentionally-drifted samples from Section 2;
a small sanity check confirms the endpoint is alive.

### 3.1 Helper: JSON invocation function

In [ ]:
def invoke_endpoint_json(endpoint_name, feature_dict):
    """Invoke endpoint with JSON format (custom handler with Athena logging)."""
    import json
    import time
    
    payload = json.dumps(feature_dict)
    start = time.time()
    
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',  # Custom handler expects JSON
        Body=payload
    )
    
    latency_ms = (time.time() - start) * 1000
    result = json.loads(response['Body'].read().decode())
    
    # Custom handler returns: {"predictions": [0], "probabilities": {"fraud": [0.1234], "non_fraud": [0.8766]}}
    fraud_prob = result["probabilities"]["fraud"][0]
    prediction = result["predictions"][0]
    
    return {'prediction': prediction, 'fraud_probability': fraud_prob}, latency_ms


### 3.2 Sanity check: one hard-coded payload

In [ ]:
# Single non-drifted prediction — uses a hardcoded payload, not Athena data.
import json

sanity_payload = {
    "transaction_hour": 14, "transaction_day_of_week": 2,
    "transaction_amount": 149.62, "transaction_type_code": 1,
    "customer_age": 42, "customer_gender": 0,
    "customer_tenure_months": 36, "account_age_days": 1095,
    "distance_from_home_km": 5.2, "distance_from_last_transaction_km": 2.3,
    "time_since_last_transaction_min": 120,
    "online_transaction": 1, "international_transaction": 0,
    "high_risk_country": 0, "merchant_category_code": 5411,
    "merchant_reputation_score": 0.85, "chip_transaction": 1,
    "pin_used": 1, "card_present": 1, "cvv_match": 1,
    "address_verification_match": 1,
    "num_transactions_24h": 3, "num_transactions_7days": 12,
    "avg_transaction_amount_30days": 125.50,
    "max_transaction_amount_30days": 450.00,
    "velocity_score": 0.3, "recurring_transaction": 0,
    "previous_fraud_incidents": 0,
    "credit_limit": 5000.0, "available_credit_ratio": 0.75,
}

result, latency_ms = invoke_endpoint_json(ENDPOINT_NAME, sanity_payload)
print(f"Prediction       : {result['prediction']}")
print(f"Fraud probability: {result['fraud_probability']:.4f}")
print(f"Latency          : {latency_ms:.0f} ms")
print(f"\nLogged to {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE} (visible after SQS flush).")


### 3.3 Bulk drifted inferences

In [ ]:
# Bulk drifted inferences — requires Section 2 to have generated the drifted CSV.
NUM_BULK_TESTS = 101

if not CSV_DRIFTED_DATA.exists():
    print(f'⚠ Drifted CSV not found: {CSV_DRIFTED_DATA}')
    print('  Run Section 2 first to generate it, then re-run this cell.')
else:
    drifted_df = pd.read_csv(CSV_DRIFTED_DATA)
    for col in TRAINING_FEATURES:
        if col in drifted_df.columns:
            drifted_df[col] = pd.to_numeric(drifted_df[col], errors='coerce')
    drifted_df = drifted_df.dropna(subset=TRAINING_FEATURES, how='all')

    n = min(NUM_BULK_TESTS, len(drifted_df))
    samples = drifted_df.sample(n=n, random_state=123)

    print(f'Source           : {CSV_DRIFTED_DATA.name}')
    print(f'Sending          : {n} rows to {ENDPOINT_NAME}')
    print()

    predictions, latencies, errors = [], [], []
    for _, row in samples.iterrows():
        try:
            result, lat = invoke_endpoint_json(ENDPOINT_NAME, row[TRAINING_FEATURES].to_dict())
            predictions.append(result)
            latencies.append(lat)
        except Exception as e:
            errors.append(str(e))

    lat_arr = np.array(latencies) if latencies else np.array([0])
    fraud = sum(1 for p in predictions if p['prediction'] == 1)
    print(f'Sent             : {len(predictions)}/{n}  ({len(errors)} errors)')
    print(f'Latency P50/P95/P99: {np.percentile(lat_arr, 50):.0f} / {np.percentile(lat_arr, 95):.0f} / {np.percentile(lat_arr, 99):.0f} ms')
    print(f'Fraud predictions: {fraud}/{len(predictions)} ({fraud / max(len(predictions), 1) * 100:.1f}%)')


### 3.4 Verify predictions landed in Athena

In [ ]:
# Verify predictions landed in Athena.
query = f"""
SELECT COUNT(*) AS total,
       CAST(MIN(request_timestamp) AS TIMESTAMP(3)) AS earliest,
       CAST(MAX(request_timestamp) AS TIMESTAMP(3)) AS latest
FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
WHERE endpoint_name = '{ENDPOINT_NAME}'
"""
result = athena_client.execute_query(query)
if not result.empty and result['total'].iloc[0] > 0:
    print(f"✓ Found {result['total'].iloc[0]:,} predictions for {ENDPOINT_NAME}")
    print(f"  Time range: {result['earliest'].iloc[0]} → {result['latest'].iloc[0]}")
else:
    print('⚠ No predictions found yet — wait ~30s for the SQS→Lambda flush, then retry.')


## 4. Apply Ground Truth

In production, labels arrive asynchronously from fraud investigations. For
dev/test we simulate them here so model-drift metrics have something to score.

**Do not re-run Section 3 after this point** — newly logged predictions would
have NULL ground_truth and wouldn't contribute to model drift until the
simulator runs again.

### 4.1 Simulator parameters

In [ ]:
# Ground truth simulation parameters from src/config/config.yaml → ground_truth_simulation.
# Only two knobs reduce simulated accuracy:
#   effective_accuracy = max(0.5, base_accuracy
#                                 - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT
#                                 - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'Base accuracy        : {GROUND_TRUTH_SIM_ACCURACY:.2f}')
print(f'Feature-drift impact : -{GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT:.2f}')
print(f'Model-drift magnitude: -{GROUND_TRUTH_SIM_MODEL_DRIFT_MAG:.2f}')
effective = max(0.5, GROUND_TRUTH_SIM_ACCURACY - GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT - GROUND_TRUTH_SIM_MODEL_DRIFT_MAG)
print(f'\nEffective accuracy   : {effective:.2f}  → ~{int((1 - effective) * 100)}% of labels will be flipped')


### 4.2 Generate and apply simulated labels

In [ ]:
# SQS → Lambda inference logging is async (CFN-deployed InferenceLoggerFunction
# batches messages: flushes every 10 records OR every 30 seconds, whichever first).
# After Section 3.3's bulk test, wait one full flush cycle so the LAST record's
# time-based flush has fired — otherwise the simulator's WHERE ground_truth IS NULL
# query will miss those rows and they'll stay unlabeled until the next sim run.
import time
print('Waiting 60s for the final SQS→Lambda flush cycle...')
time.sleep(60)

# Option 2: Simulate ground truth programmatically (in notebook)
from src.drift_monitoring.simulate_ground_truth_from_athena import GroundTruthSimulator

# Create simulator with configured drift parameters
simulator = GroundTruthSimulator(
    athena_client=athena_client,
    endpoint_name=ENDPOINT_NAME,
    accuracy=GROUND_TRUTH_SIM_ACCURACY,
    fraud_confirmation_days=(1, 7),
    non_fraud_confirmation_days=(1, 30),
    feature_drift_impact=GROUND_TRUTH_SIM_FEATURE_DRIFT_IMPACT,
    model_drift_magnitude=GROUND_TRUTH_SIM_MODEL_DRIFT_MAG,
    seed=42
)

# Run simulation
print("Simulating ground truth for predictions without ground truth...")
stats = simulator.simulate_and_write(limit=None)  # Set limit=100 to only process 100 predictions

print("\n" + "=" * 80)
print("Simulation Summary")
print("=" * 80)

if stats['total_predictions'] > 0:
    print(f"Predictions processed: {stats['total_predictions']:,}")
    print(f"Ground truth updates created: {stats['updates_created']:,}")
    print(f"Actual fraud: {stats['actual_fraud']:,} ({stats['actual_fraud']/stats['total_predictions']*100:.1f}%)")
    print(f"False positives: {stats['false_positives']:,}")
    print(f"False negatives: {stats['false_negatives']:,}")
    print(f"Model accuracy: {stats.get('accuracy', GROUND_TRUTH_SIM_ACCURACY)*100:.1f}%")
    print("=" * 80)
    print("\n✅ Ground truth simulation complete!")
    print("   Next: Run the cell below to apply updates to inference_responses table")
else:
    print("⚠️  No predictions found to simulate ground truth")
    print("   Total predictions: 0")
    print("   Possible reasons:")
    print("   - No inference tests have been run yet (run Cells 6-9)")
    print("   - All predictions already have ground truth")
    print("   - Wrong endpoint name filter")
    print("\n   Action: Run Cells 6-9 to generate inference predictions first")
    print("=" * 80)


In [ ]:
# Apply simulated ground truth updates to inference_responses table
print("Applying ground truth updates to inference_responses table...")

# Initialize updater (already imported above)
if 'updater' not in dir():
    from src.drift_monitoring.update_ground_truth import GroundTruthUpdater
    updater = GroundTruthUpdater(athena_client=athena_client, dry_run=False)

# Get statistics before update
print("\nGetting update statistics...")
update_stats = updater.get_update_statistics()

if update_stats['total_updates'] > 0:
    print(f"\nPending updates: {update_stats['total_updates']:,}")
    print(f"  Fraud cases: {update_stats['fraud_cases']:,}")
    print(f"  False positives: {update_stats['false_positives']:,}")
    print(f"  False negatives: {update_stats['false_negatives']:,}")
    print(f"  Avg days to confirmation: {update_stats['avg_days_to_confirmation']:.1f}")
    
    # Execute the update
    print("\nMerging ground truth updates into inference_responses...")
    result = updater.update_ground_truth_batch()
    
    print(f"\n✅ Ground truth update complete!")
    print(f"   Records updated: {result['updated']:,}")
    print(f"\n   Next: Run the cell below to check coverage")
else:
    print("\n⚠️  No pending updates found")
    print("   Make sure you ran the simulation cell above first")

### 4.3 Verify coverage

In [ ]:
# Check current ground truth coverage before update
from src.drift_monitoring.update_ground_truth import GroundTruthUpdater

updater = GroundTruthUpdater(athena_client=athena_client, dry_run=False)

coverage_before = updater.get_coverage_statistics()
print('Current ground truth coverage:')
print(f"  Total predictions: {coverage_before['total_predictions']:,}")
print(f"  With ground truth: {coverage_before['with_ground_truth']:,} ({coverage_before['coverage_pct']:.2f}%)")
print(f"  Without ground truth: {coverage_before['without_ground_truth']:,}")

## 5. Detect Drift

The core of the notebook. Two independent checks, both traceable to the exact
model + data snapshot that produced the predictions (lineage pinned in
`baseline.json` on the registered ModelPackage):

- **5.2 Data drift** — per-feature Kolmogorov-Smirnov test (Evidently
  `DataDriftPreset`): `training_data` baseline vs. `inference_responses` traffic.
- **5.3 Model drift** — classification-metric drift (Evidently
  `ClassificationPreset`): `evaluation_data` baseline vs. labeled
  `inference_responses`.

Results are logged to MLflow and the `monitoring_responses` Athena table (5.4),
which is exactly what the scheduled Lambda in Section 6 writes too.

### 5.0 Run setup — `monitoring_run_id` + baseline lineage

Each pass produces one `monitoring_run_id`, reused by 5.2 / 5.3 / 5.4. This cell
also finds the last monitoring run so the drift checks scope the "current"
window to only predictions since then (re-running measures *new* drift, not all
cumulative traffic).

In [ ]:
# Resolve run id + lineage + the "since when" cutoff for the current window.
from src.drift_monitoring.lambda_drift_monitor import load_baseline_from_registry
from datetime import datetime as _dt

monitoring_run_id = f'notebook-drift-{_dt.now().strftime("%Y%m%d-%H%M%S")}'
monitoring_run_ts = _dt.now()

# --- Baseline lineage (single lookup, reused by 6.2 / 6.3 / 6.4) ---
_baseline_meta = load_baseline_from_registry() or {}
model_package_arn      = _baseline_meta.get('model_package_arn') or '(not pinned)'
training_table         = _baseline_meta.get('training_table')   or ATHENA_TRAINING_TABLE
training_snapshot_id   = _baseline_meta.get('training_snapshot_id') or ''
evaluation_table       = _baseline_meta.get('evaluation_table') or ATHENA_EVALUATION_TABLE
evaluation_snapshot_id = _baseline_meta.get('evaluation_snapshot_id') or ''

# --- Previous monitoring timestamp (delta filter for the "current" window) ---
prev_ts_query = f"""
    SELECT MAX(monitoring_timestamp) AS prev_ts
    FROM {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}
    WHERE endpoint_name = '{ENDPOINT_NAME}'
"""
try:
    _prev = athena_client.execute_query(prev_ts_query)
    if _prev.empty:
        prev_monitoring_ts = None
    else:
        _val = _prev['prev_ts'].iloc[0]
        # pandas converts SQL NULL TIMESTAMP -> pd.NaT, which is NOT None
        # but ALSO can't be formatted as a TIMESTAMP literal. Treat both as cold-start.
        prev_monitoring_ts = None if pd.isna(_val) else _val
except Exception:
    # First-ever run: table empty / not yet readable. Cold-start fallback.
    prev_monitoring_ts = None

def _from(table, snapshot_id):
    return (f'{ATHENA_DATABASE}.{table} FOR VERSION AS OF {snapshot_id}'
            if snapshot_id else f'{ATHENA_DATABASE}.{table}')

def _snap_log(snapshot_id):
    return f'Iceberg snapshot {snapshot_id}' if snapshot_id else 'live table (no snapshot pinned)'

print('━' * 72)
print('MONITORING RUN — LINEAGE')
print('━' * 72)
print(f'  monitoring_run_id   : {monitoring_run_id}')
print(f'  Endpoint            : {ENDPOINT_NAME}')
print(f'  Model package ARN   : {model_package_arn}')
print(f'  Training baseline   : {ATHENA_DATABASE}.{training_table}  ({_snap_log(training_snapshot_id)})')
print(f'  Evaluation baseline : {ATHENA_DATABASE}.{evaluation_table}  ({_snap_log(evaluation_snapshot_id)})')
if prev_monitoring_ts is not None:
    print(f'  Current window      : predictions since last run @ {prev_monitoring_ts}')
else:
    print(f'  Current window      : cold-start fallback (last {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days for data drift, '
          f'last 30 days for model drift)')
print('━' * 72)


### 5.1 Overall classification metrics

`ModelPerformanceMonitor` computes precision/recall/F1/ROC-AUC against
`ground_truth` — the "current" half of the model-drift comparison, printed
inline before the detailed Evidently report.

In [ ]:
from src.drift_monitoring.monitor_model_performance import ModelPerformanceMonitor

monitor = ModelPerformanceMonitor(
    athena_client=athena_client,
    alert_threshold=0.85,  # ROC-AUC threshold for alerts
    min_samples=50,        # Minimum samples for reliable metrics
)

# Check ground truth coverage
coverage = monitor.get_ground_truth_coverage(endpoint_name=ENDPOINT_NAME, days=30)
print(f"Ground truth coverage (last 30 days):")
print(f"  Total predictions: {coverage['total_predictions']:,}")
print(f"  With ground truth: {coverage['with_ground_truth']:,} ({coverage['coverage_pct']:.2f}%)")

In [ ]:
# Generate performance report
report = monitor.generate_performance_report(
    endpoint_name=ENDPOINT_NAME,
    days=7,
    window='D',  # Daily time windows
)

monitor.print_report_summary(report)

In [ ]:
# Display overall metrics
if report.get('overall_metrics') and 'error' not in report['overall_metrics']:
    metrics = report['overall_metrics']
    print('Overall Model Performance:')
    print(f"  ROC-AUC:    {metrics.get('roc_auc', 'N/A')}")
    print(f"  PR-AUC:     {metrics.get('pr_auc', 'N/A')}")
    print(f"  Precision:  {metrics['precision']:.4f}")
    print(f"  Recall:     {metrics['recall']:.4f}")
    print(f"  F1-Score:   {metrics['f1_score']:.4f}")
    print(f"  Accuracy:   {metrics['accuracy']:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"  TP: {metrics['true_positives']:,}  FP: {metrics['false_positives']:,}")
    print(f"  FN: {metrics['false_negatives']:,}  TN: {metrics['true_negatives']:,}")
else:
    print('Insufficient ground truth data for metrics. Run ground truth update first.')

### 5.2 Data drift

Per-feature KS test (Evidently `DataDriftPreset`). Baseline = `training_data`
(the distribution the model was trained on); current = `inference_responses`
within the window resolved in 5.0. `customer_gender` is excluded (string in
training data vs. encoded float in inference — type mismatch).

In [ ]:
import json as _json
from src.drift_monitoring.evidently_reports import run_data_drift_report

# ════════════════════════════════════════════════════════════════════════
# DATA DRIFT — feature-distribution shift (Evidently DataDriftPreset)
# ════════════════════════════════════════════════════════════════════════
# Baseline : `training_data` Iceberg slice (the distribution the model was
#            TRAINED on — this is the industry-standard reference for data
#            drift: did production traffic move away from what we taught it?).
#            Lambda also uses training_data after this redesign.
# Current  : `inference_responses` rows since the last monitoring run
#            (or the last MONITORING_DATA_DRIFT_LOOKBACK_DAYS as cold-start).
# Method   : per-feature KS-test (numerical), p-value < 0.05 → drifted.
# Excluded : `customer_gender` (STRING in training_data, encoded float in
#            inference_responses — can't run KS across the type mismatch).

BASELINE_LIMIT = 5000
INFERENCE_LIMIT = 10000
NUMERIC_FEATURES = [f for f in TRAINING_FEATURES if f != 'customer_gender']

baseline_from = _from(training_table, training_snapshot_id)

# Current window: since last monitoring run if available, else cold-start lookback
if prev_monitoring_ts is not None:
    current_filter = f"request_timestamp > TIMESTAMP '{prev_monitoring_ts}'"
    window_label = f'since last monitoring run @ {prev_monitoring_ts}'
else:
    cold_start = (_dt.now() - timedelta(days=MONITORING_DATA_DRIFT_LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')
    current_filter = f"request_timestamp >= TIMESTAMP '{cold_start}'"
    window_label = f'cold-start fallback (last {MONITORING_DATA_DRIFT_LOOKBACK_DAYS} days)'

print(f'Baseline source     : {ATHENA_DATABASE}.{training_table}  ({_snap_log(training_snapshot_id)})')
print(f'Current window      : {window_label}')
print(f'Features compared   : {len(NUMERIC_FEATURES)} numeric (customer_gender excluded — type mismatch)')

# --- Baseline rows ---
baseline_sql = f"""
    SELECT {", ".join(NUMERIC_FEATURES)}
    FROM {baseline_from}
    WHERE is_fraud IS NOT NULL
    LIMIT {BASELINE_LIMIT}
"""
baseline_df = athena_client.execute_query(baseline_sql)
baseline_df = baseline_df.apply(pd.to_numeric, errors='coerce').dropna(how='all')
print(f'\nBaseline : {len(baseline_df):,} rows × {len(NUMERIC_FEATURES)} features')

# --- Current rows (parse JSON-encoded features) ---
inference_sql = f"""
    SELECT input_features
    FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
    WHERE endpoint_name = '{ENDPOINT_NAME}'
      AND {current_filter}
    LIMIT {INFERENCE_LIMIT}
"""
inf_raw = athena_client.execute_query(inference_sql)
print(f'Current  : {len(inf_raw):,} rows')

drift_results = None
if inf_raw.empty:
    print('\n⚠ No new inference data in the current window. Run Section 3 again, then re-run Section 6.')
elif len(inf_raw) < 50:
    print(f'\n⚠ Only {len(inf_raw)} rows — statistical tests need ≥ 50 for reliable p-values. Send more.')
else:
    current_df = pd.DataFrame([
        {f: float(d.get(f, float("nan"))) for f in NUMERIC_FEATURES}
        for d in (_json.loads(r) for r in inf_raw['input_features'])
    ]).dropna(how='all')

    valid = [c for c in NUMERIC_FEATURES
             if c in baseline_df.columns and c in current_df.columns
             and baseline_df[c].notna().any() and current_df[c].notna().any()]
    if len(valid) < len(NUMERIC_FEATURES):
        dropped = set(NUMERIC_FEATURES) - set(valid)
        print(f'  ⚠ Dropping {len(dropped)} all-NaN columns: {sorted(dropped)}')

    drift_results = run_data_drift_report(
        baseline_df=baseline_df[valid],
        current_df=current_df[valid],
    )

    print('\nRESULTS')
    print('━' * 72)
    print(f'  Drift detected     : {drift_results["drift_detected"]}')
    print(f'  Drifted features   : {drift_results["drifted_columns_count"]} / {len(valid)}'
          f' ({drift_results["drifted_columns_share"]:.1%})')
    if drift_results["drift_detected"]:
        # Rank drifted columns by drift_magnitude (test-agnostic: 1.0 = at
        # threshold, higher = more drifted). Evidently picks Wasserstein/PSI
        # or KS per column based on sample size, so raw drift_score direction
        # varies — magnitude normalizes across tests.
        top = sorted(
            (
                (c, info["drift_score"], info.get("method", "?"),
                 info.get("drift_magnitude", 0))
                for c, info in drift_results.get("per_column", {}).items()
                if info.get("drifted")
            ),
            key=lambda x: x[3],
            reverse=True,
        )[:5]
        if top:
            print('  Top 5 drifted      :')
            for c, score, method, mag in top:
                print(f'    - {c:32s}  score={score:.4f}  method={method}  (×{mag:.1f} over threshold)')
        else:
            print('  Top 5 drifted      : (aggregate says drift, but no per-column matched — check evidently_reports.py parser)')
    print('━' * 72)


In [ ]:
# Display the interactive Evidently DataDriftPreset report.
if 'drift_results' in dir() and drift_results and drift_results.get('snapshot'):
    print('📊 Evidently Data Drift Report (interactive)')
    print('=' * 80)
    display(drift_results['snapshot'])
else:
    print('No drift results to visualize — re-run the cell above.')


In [ ]:
# Per-feature KS p-values, sorted ascending (lower = more drifted).
if 'drift_results' in dir() and drift_results and drift_results.get('per_column'):
    print('📋 Per-Column Drift Summary')
    print('=' * 80)
    for col, info in sorted(drift_results['per_column'].items(), key=lambda x: x[1]['drift_score']):
        status = '⚠ DRIFT' if info['drifted'] else '✓ OK'
        print(f'  {col:35s} p-value={info["drift_score"]:.4f}  [{status}]')
else:
    print('No drift results — re-run the cell above.')


### 5.3 Model drift

Classification-metric drift (Evidently `ClassificationPreset`). Baseline
`(is_fraud, fraud_prediction)` pairs come from `evaluation_data`; current
`(ground_truth, prediction)` from labeled `inference_responses`. If you see
"Insufficient ground truth", re-run Section 4.2.

In [ ]:
from src.drift_monitoring.evidently_reports import run_classification_report

# ════════════════════════════════════════════════════════════════════════
# MODEL DRIFT — Evidently ClassificationPreset
# ════════════════════════════════════════════════════════════════════════
# Baseline : (is_fraud, fraud_prediction) from `evaluation_data` (the
#            labeled held-out test set the model was scored on at training
#            time — this is the right reference for model-drift, since
#            training_data labels are what the model already absorbed).
# Current  : (ground_truth, prediction) from `inference_responses` since
#            the last monitoring run (or last 30 days cold-start).
#
# Both datasets must contain BOTH classes (0 and 1). evaluation_data is
# ~0.2% fraud, so a flat LIMIT typically returns only class 0 → we
# stratify the baseline pull below to guarantee both labels appear.
# (See evidently_reports.run_classification_report for the bug context.)

MIN_SAMPLES = 50
MODEL_DRIFT_LOOKBACK_DAYS = 30
PREDICTION_THRESHOLD = 0.5

baseline_from = _from(evaluation_table, evaluation_snapshot_id)

# Current window: since last monitoring run, else cold-start
if prev_monitoring_ts is not None:
    days_window = MODEL_DRIFT_LOOKBACK_DAYS  # caps the load_predictions_with_ground_truth window
    window_label = f'since last monitoring run @ {prev_monitoring_ts}'
else:
    days_window = MODEL_DRIFT_LOOKBACK_DAYS
    window_label = f'cold-start fallback (last {MODEL_DRIFT_LOOKBACK_DAYS} days)'

print(f'Baseline source     : {ATHENA_DATABASE}.{evaluation_table}  ({_snap_log(evaluation_snapshot_id)})')
print(f'Current window      : {window_label}')
print(f'Decision threshold  : probability_fraud >= {PREDICTION_THRESHOLD}')

# Pull current predictions with ground truth, then apply the delta filter manually.
gt_df = monitor.load_predictions_with_ground_truth(
    endpoint_name=ENDPOINT_NAME, days=days_window,
)
if prev_monitoring_ts is not None and not gt_df.empty:
    gt_df = gt_df[gt_df['request_timestamp'] > prev_monitoring_ts]
print(f'\nCurrent  : {len(gt_df):,} predictions with ground truth')

model_report = None
if len(gt_df) < MIN_SAMPLES:
    print(f'⚠ Need ≥ {MIN_SAMPLES} predictions with ground truth — run Section 4.2 (simulator) again.')
elif gt_df['ground_truth'].nunique() < 2:
    print(f'⚠ Current sample has only one class — increase simulator accuracy degradation or send more drifted samples.')
else:
    current_df = pd.DataFrame({
        'target':     gt_df['ground_truth'].astype(int).values,
        'prediction': (gt_df['probability_fraud'] >= PREDICTION_THRESHOLD).astype(int).values,
    })

    # Stratified baseline pull
    half = max(MIN_SAMPLES // 2, len(gt_df) // 2)
    baseline_sql = f"""
        (SELECT CAST(is_fraud AS INT) AS target,
                CAST(fraud_prediction AS INT) AS prediction
         FROM {baseline_from}
         WHERE is_fraud = TRUE AND fraud_prediction IS NOT NULL
         LIMIT {half})
        UNION ALL
        (SELECT CAST(is_fraud AS INT) AS target,
                CAST(fraud_prediction AS INT) AS prediction
         FROM {baseline_from}
         WHERE is_fraud = FALSE AND fraud_prediction IS NOT NULL
         LIMIT {half})
    """
    baseline_df = athena_client.execute_query(baseline_sql)
    print(f'Baseline : {len(baseline_df):,} predictions '
          f'(stratified: {sum(baseline_df["target"] == 1)} fraud / {sum(baseline_df["target"] == 0)} non-fraud)')

    # Evidently ClassificationPreset needs BOTH classes in BOTH target AND
    # prediction of BOTH datasets. Check all four before calling; print a
    # clear skip message if any side is degenerate (model never predicted
    # the minority class, etc.).
    sides = [('baseline.target', baseline_df['target']),
             ('baseline.prediction', baseline_df['prediction']),
             ('current.target', current_df['target']),
             ('current.prediction', current_df['prediction'])]
    degenerate = [name for name, col in sides if col.nunique() < 2]
    if degenerate:
        print(f'⚠ Skipping classification report — single-class column(s): {degenerate}')
        print(f'  Likely cause: model never predicted the minority class on this sample.')
        print(f'  Fix: send more drifted inferences, or lower PREDICTION_THRESHOLD below 0.5.')
    else:
        model_report = run_classification_report(
            baseline_df=baseline_df, current_df=current_df,
            target_column='target', prediction_column='prediction',
        )
        print('\n✓ Classification report ready — interactive view below.')

if model_report:
    print()
    print('Evidently Classification Report (interactive)')
    print('=' * 80)
    display(model_report['snapshot'])


### 5.4 Log results to MLflow and Athena

Two writes keyed on the same `monitoring_run_id`: (1) an MLflow run with all
drift metrics + the Evidently HTML reports as artifacts; (2) one row in
`monitoring_responses` (with model-package ARN + snapshot id) plus a backfill
tagging the inference rows this run measured — so QuickSight can join them.

In [ ]:
from src.drift_monitoring.log_monitoring_to_mlflow import log_monitoring_to_mlflow

drift_data   = drift_results if 'drift_results' in dir() else None
model_data   = model_report  if 'model_report'  in dir() else None
metrics_data = report.get('overall_metrics') if 'report' in dir() and report.get('overall_metrics') else None

result = log_monitoring_to_mlflow(
    drift_results=drift_data,
    model_report=model_data,
    overall_metrics=metrics_data,
    endpoint_name=ENDPOINT_NAME,
    model_name=MLFLOW_MODEL_NAME,
    region=REGION,
)
if result['success']:
    print(f"✓ Logged to MLflow — run_id={result['mlflow_run_id']}")
else:
    print(f"⚠ MLflow logging failed: {result.get('error', 'unknown')}")


In [ ]:
# Write the monitoring run + tag the inference rows it measured.
#
# Two Athena ops, both keyed on the SAME monitoring_run_id (resolved in 6.0):
#   1. INSERT one row into monitoring_responses (the drift verdict for this run)
#   2. UPDATE inference_responses to backfill monitoring_run_id on the
#      predictions that fell in this run's current window
#
# Column list in op #1 MUST match the CFN DDL
# (cloudformation/sagemaker-mlflow-setup.yaml: monitoring_responses) AND
# src/drift_monitoring/deploy_monitoring_writer.py. Keep all three in sync.
import json as _json

# --- Aggregate drift findings into Athena-bound values ---
data_detected, drifted_count, drifted_share = False, 0, 0.0
features_analyzed, data_sample = 0, 0
per_feature = {}
if 'drift_results' in dir() and drift_results:
    data_detected     = drift_results.get('drift_detected', False)
    drifted_count     = drift_results.get('drifted_columns_count', 0)
    drifted_share     = drift_results.get('drifted_columns_share', 0)
    features_analyzed = drift_results.get('features_analyzed', 0)
    data_sample       = drift_results.get('sample_size', 0)
    for col, info in drift_results.get('per_column', {}).items():
        per_feature[col] = info.get('drift_score', 0)

mdrift = baseline_auc = current_auc = degrad = degrad_pct = None
acc = prec = rec = f1 = model_sample = None
if 'report' in dir() and report.get('overall_metrics') and 'error' not in report.get('overall_metrics', {}):
    om = report['overall_metrics']
    mdrift       = om.get('model_drift_detected', False)
    baseline_auc = om.get('baseline_roc_auc')
    current_auc  = om.get('current_roc_auc')
    degrad       = om.get('roc_auc_degradation')
    degrad_pct   = om.get('roc_auc_degradation_pct')
    acc          = om.get('accuracy')
    prec         = om.get('precision')
    rec          = om.get('recall')
    if prec and rec and (prec + rec) > 0:
        f1 = 2 * prec * rec / (prec + rec)
    model_sample = om.get('sample_size')

def sql_val(v):
    if v is None:                return 'NULL'
    if isinstance(v, bool):      return 'TRUE' if v else 'FALSE'
    if isinstance(v, str):       return "'" + v.replace("'", "''") + "'"
    return str(v)

mp_arn = _baseline_meta.get('model_package_arn')
# Use the model-package short-id as the model_version label so it's readable
model_version_label = mp_arn.split('/')[-1] if mp_arn else 'latest'

columns = [
    'monitoring_run_id', 'monitoring_timestamp',
    'endpoint_name', 'model_version', 'model_package_arn', 'evaluation_snapshot_id',
    'data_drift_detected', 'drifted_columns_count', 'drifted_columns_share',
    'features_analyzed', 'data_sample_size', 'model_drift_detected',
    'baseline_roc_auc', 'current_roc_auc',
    'roc_auc_degradation', 'roc_auc_degradation_pct',
    'accuracy', 'precision', 'recall', 'f1_score',
    'model_sample_size', 'per_feature_drift_scores',
    'evidently_report_s3_path', 'mlflow_run_id',
    'alert_sent', 'detection_engine', 'created_at',
]
ts = f"TIMESTAMP '{monitoring_run_ts.strftime('%Y-%m-%d %H:%M:%S')}'"
values = [
    sql_val(monitoring_run_id),
    ts,
    sql_val(ENDPOINT_NAME),
    sql_val(model_version_label),
    sql_val(mp_arn),
    sql_val(evaluation_snapshot_id or None),
    sql_val(data_detected), sql_val(drifted_count), sql_val(drifted_share),
    sql_val(features_analyzed), sql_val(data_sample), sql_val(mdrift),
    sql_val(baseline_auc), sql_val(current_auc),
    sql_val(degrad), sql_val(degrad_pct),
    sql_val(acc), sql_val(prec), sql_val(rec), sql_val(f1),
    sql_val(model_sample),
    sql_val(_json.dumps(per_feature) if per_feature else None),
    sql_val(None),                                  # evidently_report_s3_path
    sql_val(result.get('mlflow_run_id') if 'result' in dir() and isinstance(result, dict) else None),
    sql_val(data_detected or mdrift),
    sql_val('evidently'),
    ts,
]
insert_sql = (
    f"INSERT INTO {ATHENA_DATABASE}.{MONITORING_TABLE_NAME} "
    f"({', '.join(columns)}) VALUES ({', '.join(values)})"
)

# --- 1) Write the monitoring_responses row ---
try:
    athena_client.execute_query(insert_sql, return_results=False)
    print(f'✓ Wrote {monitoring_run_id} to {ATHENA_DATABASE}.{MONITORING_TABLE_NAME}')
    print(f'  Model package ARN     : {mp_arn or "(not pinned)"}')
    print(f'  Evaluation snapshot id: {evaluation_snapshot_id or "(not pinned)"}')
    print(f'  Training snapshot id  : {training_snapshot_id or "(not pinned)"}')
    print(f'  Data drift detected   : {data_detected}')
    print(f'  Model drift detected  : {mdrift}')
except Exception as e:
    print(f'✗ Failed to write monitoring_responses row: {e}')

# --- 2) Backfill monitoring_run_id on the inference rows we just measured ---
# Window is the same one cells 6.2 / 6.3 used: (prev_monitoring_ts, monitoring_run_ts].
# UPDATE only touches rows still tagged NULL so re-runs of this cell are idempotent.
window_lower = (
    f"TIMESTAMP '{prev_monitoring_ts}'" if prev_monitoring_ts is not None
    else f"TIMESTAMP '{(_dt.now() - timedelta(days=MONITORING_DATA_DRIFT_LOOKBACK_DAYS)).strftime('%Y-%m-%d %H:%M:%S')}'"
)
window_upper = f"TIMESTAMP '{monitoring_run_ts.strftime('%Y-%m-%d %H:%M:%S')}'"

backfill_sql = f"""
    UPDATE {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
    SET monitoring_run_id = '{monitoring_run_id}'
    WHERE endpoint_name = '{ENDPOINT_NAME}'
      AND monitoring_run_id IS NULL
      AND request_timestamp > {window_lower}
      AND request_timestamp <= {window_upper}
"""
try:
    athena_client.execute_query(backfill_sql, return_results=False)
    # Read back how many got tagged
    verify_sql = f"""
        SELECT COUNT(*) AS n FROM {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE}
        WHERE monitoring_run_id = '{monitoring_run_id}'
    """
    res = athena_client.execute_query(verify_sql)
    n_tagged = int(res['n'].iloc[0]) if not res.empty else 0
    print(f'✓ Tagged {n_tagged} inference rows with monitoring_run_id={monitoring_run_id}')
except Exception as e:
    print(f'⚠ Could not backfill monitoring_run_id on inference_responses: {e}')
    print('  (Schema may not include the column yet — see CFN inference_responses DDL.)')


### 5.5 Performance alerts (from this run)

Buckets where ROC-AUC dropped below `MIN_ROC_AUC_THRESHOLD` (config.yaml).
Informational here; the scheduled Lambda fires real SNS alerts on the same data.

In [ ]:
# Check for performance degradation alerts
alerts = report.get('alerts', [])

if alerts:
    print(f'⚠ {len(alerts)} performance alert(s) detected:\n')
    for alert in alerts:
        print(f"  [{alert['severity'].upper()}] Period: {alert['period']}")
        print(f"    ROC-AUC: {alert['value']:.4f} (baseline: {alert['baseline']:.4f})")
        print(f"    Degradation: {alert['degradation_pct']:.1f}%")
        print(f"    Samples: {alert['sample_count']:,}")
        print()
else:
    print('✓ No performance degradation alerts. Model is performing within thresholds.')

## 6. Schedule Automated Drift Monitoring *(optional)*

Everything above ran drift detection **once, interactively**. To run the same
checks on a daily schedule, hand the logic to
`src/drift_monitoring/lambda_drift_monitor.py`, which packages the identical
`check_data_drift()` + `check_model_drift()` flow for AWS Lambda.

The supporting plane already exists (CloudFormation created the SNS topic, SQS
results queue, writer Lambda, dashboard, alarms, and the drift-monitor IAM
role). The **only** thing left is the container image — which CloudFormation
can't build — so a single script builds/pushes it, creates the function against
the CFN-provisioned role, and wires the EventBridge daily rule.

Skip this section if you only need on-demand, in-notebook monitoring.

### 6.1 Alert configuration

In [ ]:
# Alert configuration for the daily drift Lambda.
#
# ALERT_EMAIL is the SNS subscription endpoint that receives drift alerts.
# Defaults to `nobody@example.com` so the notebook runs end-to-end out of
# the box (lets you demo the pipeline without having to configure email).
# `example.com` is an IANA-reserved no-deliver domain — the SNS subscription
# stays in PendingConfirmation forever and no real email is ever sent.
#
# 🚨 FOR ANY REAL USE: override this in .env before re-running.
from src.config.config import (
    DATA_DRIFT_THRESHOLD,
    MODEL_DRIFT_THRESHOLD,
    DRIFT_MONITOR_SCHEDULE,
)

DUMMY_EMAIL = 'nobody@example.com'
ALERT_EMAIL = os.getenv('ALERT_EMAIL', DUMMY_EMAIL)

if ALERT_EMAIL == DUMMY_EMAIL:
    print('━' * 72)
    print('🚨 ALERT_EMAIL is using the dummy default — drift alerts will NOT be delivered.')
    print('━' * 72)
    print('  The deploy will succeed and the Lambda will run, but every email')
    print('  is silently dropped because example.com is a non-deliverable domain.')
    print('  To receive real alerts, set ALERT_EMAIL in .env:')
    print('      echo "ALERT_EMAIL=you@your-company.com" >> ../.env')
    print('  Then restart the kernel and re-run from the top.')
    print('━' * 72)
elif '@' not in ALERT_EMAIL:
    raise ValueError(f"ALERT_EMAIL={ALERT_EMAIL!r} is not a valid email address.")
else:
    print(f'✓ Alert email          : {ALERT_EMAIL}')

print(f'  Data drift threshold : PSI ≥ {DATA_DRIFT_THRESHOLD}')
print(f'  Model drift threshold: {MODEL_DRIFT_THRESHOLD * 100:.0f}% degradation')
print(f'  Schedule             : {DRIFT_MONITOR_SCHEDULE}')


### 6.2 Deploy the drift-monitor Lambda (container image) + daily schedule

`scripts/deploy_lambda_container.sh` builds the image (local Docker or CodeBuild
fallback), creates/updates the Lambda using the CFN-provisioned
`fraud-detection-drift-monitor-role`, and creates the EventBridge rule on the
schedule from `config.yaml` → `drift_monitor.schedule`.

In [ ]:
# Force a fresh container build+deploy of the drift-monitor Lambda.
# (The writer Lambda, SNS, SQS, dashboard, alarms and IAM role already
#  exist — CloudFormation created them. Only the container image, which
#  CFN cannot build, is deployed here.)
REDEPLOY_LAMBDAS = True
# Deploy drift monitoring Lambda as container image
import subprocess
import os
import boto3

# Cell 7.1 already validates and warns about the dummy default — no
# additional guard here; the deploy proceeds either way.

lambda_client = boto3.client('lambda', region_name=REGION)

should_deploy = REDEPLOY_LAMBDAS
if not REDEPLOY_LAMBDAS:
    try:
        lambda_client.get_function(FunctionName=DRIFT_LAMBDA_NAME)
        print(f"✓ Lambda '{DRIFT_LAMBDA_NAME}' exists. Skipping deployment.")
        print("  (Set REDEPLOY_LAMBDAS = True to force redeployment)")
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"Lambda '{DRIFT_LAMBDA_NAME}' not found. Proceeding with deployment...")
        should_deploy = True
else:
    print("REDEPLOY_LAMBDAS enabled. Forcing redeployment...")

if should_deploy:
    print("")
    print("╔════════════════════════════════════════════════════════════════════╗")
    print("║  Deploying Drift Monitoring Lambda (Container Image)              ║")
    print("╚════════════════════════════════════════════════════════════════════╝")
    print(f"  Email: {ALERT_EMAIL}")
    print(f"  Data Drift Threshold: {DATA_DRIFT_THRESHOLD}")
    print(f"  Model Drift Threshold: {MODEL_DRIFT_THRESHOLD}")
    print("")

    result = subprocess.run(
        ['bash', 'scripts/deploy_lambda_container.sh',
         ALERT_EMAIL,
         str(DATA_DRIFT_THRESHOLD),
         str(MODEL_DRIFT_THRESHOLD)],
        cwd=project_root,
        env={**os.environ, 'AWS_REGION': REGION},
        capture_output=False,
        text=True,
    )

    if result.returncode != 0:
        print("\n❌ Deployment failed!")
        print("Check the output above for errors")
    else:
        print("\n✅ Deployment successful!")
        print("\nNext steps:")
        print("  1. Check your email for SNS subscription confirmation")
        print(f"  2. Monitor logs: aws logs tail {CLOUDWATCH_LOG_GROUP_DRIFT} --follow")
        print("  3. Test manually in the next cell")


### 6.3 Manual trigger (test)

In [ ]:
import boto3
import json

lambda_client = boto3.client('lambda', region_name=REGION)

print("🧪 Testing drift monitoring Lambda function...")
print("=" * 80)

try:
    response = lambda_client.invoke(
        FunctionName=DRIFT_LAMBDA_NAME,
        InvocationType='RequestResponse',
        LogType='Tail'
    )

    payload = json.loads(response['Payload'].read())
    
    if payload.get('statusCode') == 200:
        body = json.loads(payload['body'])
        
        print("✅ Drift monitoring check completed successfully")
        print("")
        print(f"Timestamp: {body['timestamp']}")
        print("")
        
        # Data drift results
        if body.get('data_drift'):
            dd = body['data_drift']
            print("📊 DATA DRIFT RESULTS:")
            print(f"  Features Analyzed: {dd.get('features_analyzed', 'N/A')}")
            print(f"  Drifted Features: {dd.get('drifted_features_count', 'N/A')}")
            print(f"  Drift Percentage: {dd.get('drift_percentage', 0):.1f}%")
            print(f"  Average PSI: {dd.get('avg_psi', 0):.4f}")
            print(f"  Max PSI: {dd.get('max_psi', 0):.4f}")
            print(f"  Status: {'🚨 DRIFT DETECTED' if dd.get('detected') else '✓ No drift'}")
            
            if dd.get('drifted_features'):
                print("\n  Top Drifted Features:")
                for feat in dd['drifted_features']:
                    print(f"    - {feat['feature']}: drift_score={feat['drift_score']:.4f}")
        else:
            print("⚠️ Data drift: Insufficient samples")
        
        print("")
        
        # Model drift results
        if body.get('model_drift'):
            md = body['model_drift']
            print("🎯 MODEL DRIFT RESULTS:")
            print(f"  Baseline ROC-AUC: {md.get('baseline_roc_auc', 0):.4f}")
            print(f"  Current ROC-AUC: {md.get('current_roc_auc', 0):.4f}")
            print(f"  Degradation: {md.get('degradation', 0):.4f} ({md.get('degradation_pct', 0):.1f}%)")
            print(f"  Status: {'🚨 DRIFT DETECTED' if md.get('detected') else '✓ No drift'}")
        else:
            print("⚠️ Model drift: Insufficient ground truth samples")
        
        print("")
        print(f"Alert Sent: {'Yes' if body.get('alert_sent') else 'No'}")
        
    else:
        print(f"❌ Lambda returned error: {payload}")

except Exception as e:
    print(f"❌ Error invoking Lambda: {e}")
    import traceback
    traceback.print_exc()

### 6.4 View CloudWatch logs

In [ ]:
import boto3
from datetime import datetime

logs_client = boto3.client('logs', region_name=REGION)
LOG_GROUP = CLOUDWATCH_LOG_GROUP_DRIFT

print('Recent Lambda execution logs:')
print('=' * 80)

try:
    streams = logs_client.describe_log_streams(
        logGroupName=LOG_GROUP,
        orderBy='LastEventTime',
        descending=True,
        limit=1
    )

    if streams['logStreams']:
        stream_name = streams['logStreams'][0]['logStreamName']
        print(f'Latest log stream: {stream_name}')
        print()

        events = logs_client.get_log_events(
            logGroupName=LOG_GROUP,
            logStreamName=stream_name,
            limit=50
        )

        for event in events['events']:
            ts = datetime.fromtimestamp(event['timestamp'] / 1000).strftime('%H:%M:%S')
            msg = event['message'].rstrip()
            print(f'[{ts}] {msg}')
    else:
        print('No logs found yet. Run the test cell above to generate logs.')

except logs_client.exceptions.ResourceNotFoundException:
    print(f'Log group {LOG_GROUP} not found. Deploy and test the Lambda first.')
except Exception as e:
    print(f'Error fetching logs: {e}')

### 6.5 Disable / re-enable the schedule

In [ ]:
import boto3

events = boto3.client('events', region_name=REGION)

# Set to 'ENABLED' or 'DISABLED'
DESIRED_STATE = 'ENABLED'  # Change to 'DISABLED' to stop monitoring

try:
    if DESIRED_STATE == 'DISABLED':
        events.disable_rule(Name=EVENTBRIDGE_RULE_NAME)
        print("⏸️ Drift monitoring DISABLED")
    else:
        events.enable_rule(Name=EVENTBRIDGE_RULE_NAME)
        print("▶️ Drift monitoring ENABLED")
        
    # Show current status
    rule = events.describe_rule(Name=EVENTBRIDGE_RULE_NAME)
    print(f"\nCurrent status: {rule['State']}")
    print(f"Schedule: {rule['ScheduleExpression']}")
    
except Exception as e:
    print(f"❌ Error: {e}")

## Teardown

Cleanup lives in `6_optional_cleanup.ipynb`. To remove the drift-monitoring
plane created here:

```bash
# 1. Out-of-band resources first (container Lambda, EventBridge rule)
./scripts/delete_infrastructure.sh --execute
# 2. Then the CloudFormation stack (SNS, SQS, writer Lambda, dashboard, alarms, role)
aws cloudformation delete-stack --stack-name fraud-detection-drift-monitoring
```

Always delete the out-of-band drift Lambda before the stack — it references the
CFN-owned SQS queue, SNS topic, and IAM role.

---